# Sales Data — Full Python Analysis Project

**Objective:** Perform a complete business-oriented analysis of the Sales Data Excel file using Python.

### Analysis areas
- Data loading, validation and cleaning
- KPI / business summary
- Sales and quantity trends
- Product performance
- Customer analysis
- State / geographic analysis
- Sales channel analysis
- Payment type analysis
- Price and quantity analysis
- Monthly / yearly trends
- Top & bottom performers
- Correlation and distribution analysis
- Business charts
- Automated insights and recommendations

> **Input file:** `D:\PowerBI\DataSets\Sales Data.xlsx`


In [ ]:
# 1. Install/import required libraries
# If any library is missing, uncomment the relevant pip command and run it.

# %pip install pandas openpyxl numpy matplotlib seaborn

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries loaded successfully.")


In [ ]:
# 2. Load Excel file

FILE_PATH = r"D:\PowerBI\DataSets\Sales Data.xlsx"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"File not found: {FILE_PATH}\n"
        "Please verify the path or change FILE_PATH to the actual location."
    )

df = pd.read_excel(FILE_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())


## 3. Data Understanding & Data Quality

In [ ]:
# Basic structure
display(df.info())
print("\nShape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())


In [ ]:
# Standardize column names
df.columns = (
    df.columns
      .str.strip()
      .str.replace(r"\s+", " ", regex=True)
)

# Expected columns from the provided schema
expected_columns = [
    "Transaction ID", "Product", "Quantity", "Unit Price (INR)", "Date",
    "Customer Name", "State", "Country", "Sales Channel", "Payment Type", "Sales"
]

missing_expected = [c for c in expected_columns if c not in df.columns]
print("Missing expected columns:", missing_expected if missing_expected else "None")


In [ ]:
# Data types and missing values
display(df.dtypes.to_frame("Data Type"))
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

quality = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
})
display(quality[quality["Missing Count"] > 0])


In [ ]:
# Clean / convert important fields

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

numeric_cols = ["Quantity", "Unit Price (INR)", "Sales"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Calculate sales from quantity × unit price where Sales is missing
missing_sales = df["Sales"].isna()
df.loc[missing_sales, "Sales"] = (
    df.loc[missing_sales, "Quantity"] * df.loc[missing_sales, "Unit Price (INR)"]
)

# Derived columns
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month Name"] = df["Date"].dt.strftime("%b")
df["Year-Month"] = df["Date"].dt.to_period("M").astype(str)
df["Day"] = df["Date"].dt.day
df["Day Name"] = df["Date"].dt.day_name()
df["Average Selling Price"] = df["Sales"] / df["Quantity"].replace(0, np.nan)

print("Cleaning and derived columns completed.")
display(df.head())


In [ ]:
# Duplicates and invalid values
duplicate_count = df.duplicated().sum()
transaction_duplicates = df["Transaction ID"].duplicated().sum()

print("Duplicate full rows:", duplicate_count)
print("Duplicate Transaction IDs:", transaction_duplicates)
print("Negative quantity:", (df["Quantity"] < 0).sum())
print("Negative sales:", (df["Sales"] < 0).sum())
print("Zero quantity:", (df["Quantity"] == 0).sum())
print("Zero sales:", (df["Sales"] == 0).sum())
print("Invalid dates:", df["Date"].isna().sum())


## 4. Executive KPI Dashboard

In [ ]:
total_sales = df["Sales"].sum()
total_quantity = df["Quantity"].sum()
total_transactions = df["Transaction ID"].nunique()
unique_customers = df["Customer Name"].nunique()
unique_products = df["Product"].nunique()
unique_states = df["State"].nunique()
avg_transaction_value = total_sales / total_transactions if total_transactions else 0
avg_unit_price = df["Unit Price (INR)"].mean()

kpis = pd.DataFrame({
    "KPI": [
        "Total Sales", "Total Quantity", "Transactions", "Unique Customers",
        "Unique Products", "States", "Average Transaction Value",
        "Average Unit Price"
    ],
    "Value": [
        total_sales, total_quantity, total_transactions, unique_customers,
        unique_products, unique_states, avg_transaction_value, avg_unit_price
    ]
})

display(kpis)


In [ ]:
# Date coverage
print("Minimum date:", df["Date"].min())
print("Maximum date:", df["Date"].max())
print("Number of days covered:", (df["Date"].max() - df["Date"].min()).days + 1)


## 5. Sales Trend Analysis

In [ ]:
monthly_sales = (
    df.groupby("Year-Month", as_index=False)["Sales"]
      .sum()
      .sort_values("Year-Month")
)

plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_sales, x="Year-Month", y="Sales", marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales (INR)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Yearly sales
yearly_sales = df.groupby("Year", as_index=False).agg(
    Sales=("Sales", "sum"),
    Quantity=("Quantity", "sum"),
    Transactions=("Transaction ID", "nunique")
)

yearly_sales["YoY Growth %"] = yearly_sales["Sales"].pct_change() * 100
display(yearly_sales)


In [ ]:
# Monthly seasonality
monthly_seasonality = (
    df.groupby("Month", as_index=False)
      .agg(Sales=("Sales", "sum"), Quantity=("Quantity", "sum"))
)

plt.figure(figsize=(12, 5))
sns.barplot(data=monthly_seasonality, x="Month", y="Sales")
plt.title("Sales by Calendar Month")
plt.xlabel("Month Number")
plt.ylabel("Sales (INR)")
plt.tight_layout()
plt.show()


## 6. Product Performance Analysis

In [ ]:
product_perf = (
    df.groupby("Product", as_index=False)
      .agg(
          Sales=("Sales", "sum"),
          Quantity=("Quantity", "sum"),
          Transactions=("Transaction ID", "nunique"),
          Avg_Unit_Price=("Unit Price (INR)", "mean")
      )
      .sort_values("Sales", ascending=False)
)

product_perf["Sales Share %"] = product_perf["Sales"] / total_sales * 100

display(product_perf)


In [ ]:
top_n = 10

plt.figure(figsize=(12, 6))
top_products = product_perf.head(top_n).sort_values("Sales")
sns.barplot(data=top_products, x="Sales", y="Product")
plt.title(f"Top {top_n} Products by Sales")
plt.xlabel("Sales (INR)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


In [ ]:
# Product quantity vs sales
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=product_perf,
    x="Quantity",
    y="Sales",
    size="Transactions",
    sizes=(50, 500),
    legend=False
)
plt.title("Product Quantity vs Sales")
plt.xlabel("Quantity Sold")
plt.ylabel("Sales (INR)")
plt.tight_layout()
plt.show()


## 7. Customer Analysis

In [ ]:
customer_perf = (
    df.groupby("Customer Name", as_index=False)
      .agg(
          Sales=("Sales", "sum"),
          Quantity=("Quantity", "sum"),
          Transactions=("Transaction ID", "nunique"),
          Avg_Order_Value=("Sales", "mean")
      )
      .sort_values("Sales", ascending=False)
)

customer_perf["Customer Share %"] = customer_perf["Sales"] / total_sales * 100

display(customer_perf.head(20))


In [ ]:
# Customer frequency segmentation
customer_segment = customer_perf.copy()

customer_segment["Segment"] = np.select(
    [
        (customer_segment["Transactions"] >= 5) & (customer_segment["Sales"] >= customer_segment["Sales"].quantile(0.75)),
        (customer_segment["Transactions"] >= 3),
        (customer_segment["Transactions"] >= 2),
    ],
    ["High Value / Repeat", "Regular", "Repeat"],
    default="One-Time / Low Frequency"
)

display(customer_segment["Segment"].value_counts().to_frame("Customers"))


In [ ]:
# Top customers
top_customers = customer_perf.head(10).sort_values("Sales")

plt.figure(figsize=(12, 6))
sns.barplot(data=top_customers, x="Sales", y="Customer Name")
plt.title("Top 10 Customers by Sales")
plt.xlabel("Sales (INR)")
plt.ylabel("Customer")
plt.tight_layout()
plt.show()


## 8. State / Geographic Analysis

In [ ]:
state_perf = (
    df.groupby("State", as_index=False)
      .agg(
          Sales=("Sales", "sum"),
          Quantity=("Quantity", "sum"),
          Transactions=("Transaction ID", "nunique"),
          Customers=("Customer Name", "nunique")
      )
      .sort_values("Sales", ascending=False)
)

state_perf["Sales Share %"] = state_perf["Sales"] / total_sales * 100
display(state_perf)


In [ ]:
plt.figure(figsize=(12, 8))
top_states = state_perf.head(15).sort_values("Sales")
sns.barplot(data=top_states, x="Sales", y="State")
plt.title("Top 15 States by Sales")
plt.xlabel("Sales (INR)")
plt.ylabel("State")
plt.tight_layout()
plt.show()


## 9. Sales Channel Analysis

In [ ]:
channel_perf = (
    df.groupby("Sales Channel", as_index=False)
      .agg(
          Sales=("Sales", "sum"),
          Quantity=("Quantity", "sum"),
          Transactions=("Transaction ID", "nunique"),
          Customers=("Customer Name", "nunique")
      )
)

channel_perf["Sales Share %"] = channel_perf["Sales"] / total_sales * 100
channel_perf["Avg Transaction Value"] = channel_perf["Sales"] / channel_perf["Transactions"]

display(channel_perf)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=channel_perf, x="Sales Channel", y="Sales", ax=axes[0])
axes[0].set_title("Sales by Channel")
axes[0].set_xlabel("")
axes[0].set_ylabel("Sales (INR)")

sns.barplot(data=channel_perf, x="Sales Channel", y="Avg Transaction Value", ax=axes[1])
axes[1].set_title("Average Transaction Value by Channel")
axes[1].set_xlabel("")
axes[1].set_ylabel("Average Transaction Value (INR)")

plt.tight_layout()
plt.show()


## 10. Payment Type Analysis

In [ ]:
payment_perf = (
    df.groupby("Payment Type", as_index=False)
      .agg(
          Sales=("Sales", "sum"),
          Transactions=("Transaction ID", "nunique"),
          Quantity=("Quantity", "sum")
      )
)

payment_perf["Sales Share %"] = payment_perf["Sales"] / total_sales * 100
payment_perf["Avg Transaction Value"] = payment_perf["Sales"] / payment_perf["Transactions"]

display(payment_perf)


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=payment_perf.sort_values("Sales", ascending=False),
            x="Payment Type", y="Sales")
plt.title("Sales by Payment Type")
plt.xlabel("")
plt.ylabel("Sales (INR)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 11. Quantity & Pricing Analysis

In [ ]:
print("Quantity statistics:")
display(df["Quantity"].describe())

print("Unit price statistics:")
display(df["Unit Price (INR)"].describe())

print("Sales statistics:")
display(df["Sales"].describe())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Sales"], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Sales Distribution")
axes[0].set_xlabel("Sales (INR)")

sns.histplot(df["Quantity"], bins=20, kde=True, ax=axes[1])
axes[1].set_title("Quantity Distribution")
axes[1].set_xlabel("Quantity")

plt.tight_layout()
plt.show()


In [ ]:
# Outlier detection using IQR
def iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return mask, lower, upper

sales_outliers, sales_lower, sales_upper = iqr_outliers(df["Sales"])

print(f"Sales outliers: {sales_outliers.sum():,}")
print(f"Sales lower bound: {sales_lower:,.2f}")
print(f"Sales upper bound: {sales_upper:,.2f}")

display(
    df.loc[sales_outliers,
           ["Transaction ID", "Product", "Quantity", "Unit Price (INR)", "Sales",
            "Customer Name", "State"]]
    .sort_values("Sales", ascending=False)
    .head(20)
)


## 12. Correlation Analysis

In [ ]:
corr_cols = ["Quantity", "Unit Price (INR)", "Sales"]
corr = df[corr_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

display(corr)


## 13. Cross-Analysis

In [ ]:
# Product × Channel
product_channel = pd.pivot_table(
    df,
    index="Product",
    columns="Sales Channel",
    values="Sales",
    aggfunc="sum",
    fill_value=0
)

display(product_channel.sort_values(by=product_channel.columns.tolist(), ascending=False).head(15))


In [ ]:
# State × Channel
state_channel = pd.pivot_table(
    df,
    index="State",
    columns="Sales Channel",
    values="Sales",
    aggfunc="sum",
    fill_value=0
)

display(state_channel.head(20))


In [ ]:
# Product sales by year
product_year = pd.pivot_table(
    df,
    index="Product",
    columns="Year",
    values="Sales",
    aggfunc="sum",
    fill_value=0
)

display(product_year)


## 14. Automated Business Insights

In [ ]:
# Generate data-driven insights

insights = []

# Overall
insights.append(
    f"Total sales are ₹{total_sales:,.0f} across {total_transactions:,} transactions "
    f"and {unique_customers:,} unique customers."
)

# Top product
top_product = product_perf.iloc[0]
insights.append(
    f"Top product by sales is '{top_product['Product']}' with "
    f"₹{top_product['Sales']:,.0f}, contributing {top_product['Sales Share %']:.1f}% of total sales."
)

# Top state
top_state = state_perf.iloc[0]
insights.append(
    f"Top state by sales is '{top_state['State']}' with ₹{top_state['Sales']:,.0f} "
    f"({top_state['Sales Share %']:.1f}% of total sales)."
)

# Top channel
top_channel = channel_perf.sort_values("Sales", ascending=False).iloc[0]
insights.append(
    f"'{top_channel['Sales Channel']}' is the leading sales channel, generating "
    f"₹{top_channel['Sales']:,.0f} ({top_channel['Sales Share %']:.1f}% of sales)."
)

# Top payment
top_payment = payment_perf.sort_values("Sales", ascending=False).iloc[0]
insights.append(
    f"'{top_payment['Payment Type']}' is the largest payment type by sales, "
    f"with ₹{top_payment['Sales']:,.0f}."
)

# Highest month
best_month = monthly_sales.loc[monthly_sales["Sales"].idxmax()]
insights.append(
    f"The strongest month in the dataset is {best_month['Year-Month']} "
    f"with sales of ₹{best_month['Sales']:,.0f}."
)

# Customer concentration
top_10_customer_share = customer_perf.head(10)["Sales"].sum() / total_sales * 100
insights.append(
    f"The top 10 customers contribute approximately {top_10_customer_share:.1f}% of total sales, "
    f"indicating the level of customer concentration."
)

# Print
for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")


## 15. Business Recommendations

In [ ]:
recommendations = [
    "Focus retention and upselling campaigns on high-value repeat customers.",
    "Prioritize inventory and marketing for the highest-revenue products.",
    "Investigate low-performing products for pricing, positioning, or demand issues.",
    "Compare online vs store performance and optimize the weaker channel.",
    "Use state-level sales performance to prioritize regional marketing and distribution.",
    "Promote high-converting payment methods while reducing checkout friction.",
    "Monitor monthly seasonality to improve inventory planning and campaign timing.",
    "Investigate unusually large transactions separately because they can materially affect total revenue.",
    "Build a recurring dashboard with Sales, Transactions, Quantity, Average Order Value, and Growth KPIs."
]

for i, recommendation in enumerate(recommendations, 1):
    print(f"{i}. {recommendation}")


## 16. Final Analysis Tables

In [ ]:
# Export key analysis tables to Excel for further use

OUTPUT_FILE = "Sales_Analysis_Output.xlsx"

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    kpis.to_excel(writer, sheet_name="KPIs", index=False)
    yearly_sales.to_excel(writer, sheet_name="Yearly Sales", index=False)
    monthly_sales.to_excel(writer, sheet_name="Monthly Sales", index=False)
    product_perf.to_excel(writer, sheet_name="Product Analysis", index=False)
    customer_perf.to_excel(writer, sheet_name="Customer Analysis", index=False)
    state_perf.to_excel(writer, sheet_name="State Analysis", index=False)
    channel_perf.to_excel(writer, sheet_name="Channel Analysis", index=False)
    payment_perf.to_excel(writer, sheet_name="Payment Analysis", index=False)

print(f"Analysis workbook created: {OUTPUT_FILE}")


## 17. Project Conclusion

This notebook provides a complete business analysis workflow from raw Excel data to actionable insights.

### Suggested portfolio description
> **Sales Data Analysis — Python**
>
> An end-to-end sales analytics project using Python, Pandas, NumPy, Matplotlib and Seaborn. The project covers data cleaning, KPI analysis, sales trends, product and customer performance, geographic analysis, channel/payment analysis, outlier detection, correlation analysis and business recommendations.

### Interview talking points
- How did you validate and clean the data?
- How did you handle missing Sales values?
- Which product / state / channel generated the most revenue?
- How did you identify high-value customers?
- What trends or seasonality did you find?
- How did you detect outliers?
- What business actions would you recommend from the analysis?
